# Task 4: Optimize Portfolio Based on Forecast

MPT efficient frontier with TSLA forecast view.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loader import combine_close_prices, fetch_all_assets
from src.forecasting import fit_auto_arima, forecast_arima
from src.portfolio import (
    annualized_forecast_return_from_prices,
    build_expected_returns_vector,
    compute_covariance_matrix,
    optimize_portfolios,
    simulate_efficient_frontier,
)
from src.preprocessing import clean_asset_frame, compute_daily_returns


In [ ]:
asset_data = {t: clean_asset_frame(df) for t, df in fetch_all_assets().items()}
prices = combine_close_prices(asset_data)
returns = prices.pct_change().dropna()

train_prices = prices.loc[:'2024-12-31']
train_returns = returns.loc[:'2024-12-31']

tsla_train = train_prices['TSLA']
arima_model = fit_auto_arima(tsla_train, seasonal=True, m=5)
future_prices = forecast_arima(arima_model, 252)
tsla_expected = annualized_forecast_return_from_prices(
    current_price=float(tsla_train.iloc[-1]),
    forecast_prices=future_prices,
    horizon_days=252,
)

expected_returns = build_expected_returns_vector(train_returns, tsla_expected)
cov_matrix = compute_covariance_matrix(train_returns)
expected_returns, cov_matrix

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cov_matrix, annot=True, fmt='.2e', cmap='Blues')
plt.title('Covariance Matrix (Daily Returns)')
plt.show()

In [ ]:
results = optimize_portfolios(expected_returns, cov_matrix)
frontier = simulate_efficient_frontier(expected_returns, cov_matrix)

max_sharpe = results['max_sharpe']
min_vol = results['min_volatility']
max_sharpe['weights'], max_sharpe['performance']

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frontier['Volatility'], frontier['Expected Return'], label='Efficient Frontier')
plt.scatter(max_sharpe['performance'][1], max_sharpe['performance'][0], color='green', s=120, label='Max Sharpe')
plt.scatter(min_vol['performance'][1], min_vol['performance'][0], color='red', s=120, label='Min Volatility')
plt.xlabel('Volatility (Risk)')
plt.ylabel('Expected Return')
plt.title('Efficient Frontier')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Recommendation

We recommend the **Maximum Sharpe Ratio** portfolio for clients seeking strong risk-adjusted returns while maintaining diversification through BND and SPY. See optimized weights and performance metrics above.